# Plot Generation

This notebook reads evaluation results from `evaluation.ipynb` and generates publication-quality plots saved to the `plots/` directory.

**Plots generated:**
1. Binary F1-score comparison (all models)
2. Macro F1-score comparison (all models)
3. Delta / gap chart: NLP model vs each baseline

> **Prerequisite:** Run `nlp_model.ipynb` then `evaluation.ipynb` first to generate `models_artifacts/outputs/evaluation_comparison.csv`.

## 1. Imports and Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.metrics import ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

# Paths — input files written by evaluation.ipynb
COMPARISON_CSV   = '../models_artifacts/outputs/evaluation_comparison.csv'
NLP_RESULTS_JSON = '../models_artifacts/outputs/nlp_model_results.json'
PLOTS_DIR = '../plots/'
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 2. Load Evaluation Data

In [ ]:
comparison = pd.read_csv(COMPARISON_CSV, index_col=0)

with open(NLP_RESULTS_JSON, 'r') as f:
    nlp_results = json.load(f)

NLP_MODEL_LABEL = 'NLP-TF-IDF-LR (Ours)'

print('Loaded comparison table:')
comparison

## 3. Plot 1 — Binary F1-Score Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['darkorange' if idx == NLP_MODEL_LABEL else 'steelblue'
          for idx in comparison.index]

bars = ax.barh(comparison.index, comparison['Binary F1'], color=colors, edgecolor='white')

for bar, val in zip(bars, comparison['Binary F1']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=10)

nlp_patch  = mpatches.Patch(color='darkorange', label='NLP Model (Ours)')
base_patch = mpatches.Patch(color='steelblue',  label='Paper Baselines')
ax.legend(handles=[nlp_patch, base_patch], loc='lower right')

ax.set_xlabel('Binary F1-Score')
ax.set_title('Figure 1 — Binary F1-Score: NLP Model vs Paper Baselines')
ax.set_xlim(0, 1.12)
plt.tight_layout()

out_path = os.path.join(PLOTS_DIR, 'fig1_binary_f1_nlp_vs_baselines.png')
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 4. Plot 2 — Macro F1-Score Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(comparison.index, comparison['Macro F1'], color=colors, edgecolor='white')

for bar, val in zip(bars, comparison['Macro F1']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=10)

ax.legend(handles=[nlp_patch, base_patch], loc='lower right')
ax.set_xlabel('Macro-Averaged F1-Score')
ax.set_title('Figure 2 — Macro F1-Score: NLP Model vs Paper Baselines')
ax.set_xlim(0, 1.12)
plt.tight_layout()

out_path = os.path.join(PLOTS_DIR, 'fig2_macro_f1_nlp_vs_baselines.png')
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 5. Plot 3 — Performance Gap: NLP Model vs Each Baseline

In [ ]:
baselines_only = comparison.drop(index=NLP_MODEL_LABEL)

delta_binary = nlp_results['binary_f1'] - baselines_only['Binary F1']
delta_macro  = nlp_results['macro_f1']  - baselines_only['Macro F1']

delta_df = pd.DataFrame({'Binary F1 Delta': delta_binary, 'Macro F1 Delta': delta_macro})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
                           ['Binary F1 Delta', 'Macro F1 Delta'],
                           ['Binary F1 Gap', 'Macro F1 Gap']):
    bar_colors = ['forestgreen' if v >= 0 else 'crimson' for v in delta_df[col]]
    bars = ax.barh(delta_df.index, delta_df[col], color=bar_colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    for bar, val in zip(bars, delta_df[col]):
        offset = 0.003 if val >= 0 else -0.003
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height() / 2,
                f'{val:+.3f}', va='center', fontsize=9)
    ax.set_xlabel('NLP model score − baseline score')
    ax.set_title(f'Figure 3 — {title}: NLP Model vs Each Baseline')

plt.tight_layout()
out_path = os.path.join(PLOTS_DIR, 'fig3_performance_gap_nlp_vs_baselines.png')
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 6. Summary of Generated Plots

In [ ]:
plot_files = sorted([
    f for f in os.listdir(PLOTS_DIR) if f.endswith('.png')
])

print('=== Plots saved to plots/ ===')
for f in plot_files:
    print(f'  {f}')